# CLIP Metrics Fix — Re-evaluation using saved models

**What this notebook fixes:**
- Bug 1 (main): `relevant_ids = {q_id}` treats the item_id string as a single-element set — so `len(relevant_ids)` is always 1. But each item_id has **multiple gallery images**. Using 1 as denominator in AP and IDCG inflates mAP and NDCG above 1.0.
- Bug 2 (secondary): `index.knn_query(query_embs, k=max_k+1)` fetches 16 results but slices only 15 in `retrieved_ids[:k]`, so the +1 does nothing useful. With the fix we fetch exactly `max_k` since query images are in the query split, not the gallery.

**What you need from Kaggle output tab:**
- `clip_finetuned.pt`
- `gallery_index_A.bin` + `gallery_meta_A.json`
- `gallery_index_B_alpha07.bin` + `gallery_meta_B_alpha07.json`
- `gallery_index_B_alpha05.bin` + `gallery_meta_B_alpha05.json`
- `gallery_index_C_alpha07.bin` + `gallery_meta_C_alpha07.json`
- `gallery_index_C_alpha05.bin` + `gallery_meta_C_alpha05.json`

Upload all of these as a Kaggle dataset (e.g. `clip-saved-outputs`) before running.

## Cell 1 — Install

In [1]:
!pip install open-clip-torch hnswlib tqdm --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00


## Cell 2 — Imports & Paths

In [2]:
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
import open_clip
import hnswlib

# ── Dataset paths (same as original notebook) ──
DATASET_ROOT  = Path("/kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset")
IMG_ROOT      = DATASET_ROOT / "img" / "img"
SPLIT_FILE    = DATASET_ROOT / "eval" / "list_eval_partition.txt"
BBOX_FILE     = DATASET_ROOT / "Anno" / "list_bbox_inshop.txt"
CAPTIONS_FILE = Path("/kaggle/input/datasets/taralsanka/blip-updated-captions/captions.json")

# ── Saved outputs from original run — upload these as a dataset ──
SAVED_DIR = Path("/kaggle/input/datasets/taralsanka/clip-saved-outputs")   # adjust dataset name if needed
FT_MODEL  = SAVED_DIR / "clip_finetuned.pt"

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Config (must match original notebook) ──
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_MODEL    = "ViT-L-14"
CLIP_PRETRAIN = "openai"
EMB_DIM       = 768
PAD           = 0.05
BATCH_SIZE    = 64
TOP_K_LIST    = [5, 10, 15]
SEEDS         = [83, 588, 527, 33]   # same seeds as original run

print(f"Device       : {DEVICE}")
print(f"Saved dir    : {SAVED_DIR.exists()}")
print(f"FT model     : {FT_MODEL.exists()}")

Device       : cuda
Saved dir    : True
FT model     : True


## Cell 3 — Helpers (seed, bbox parsing)

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def parse_split_file(path):
    with open(path) as f:
        lines = f.readlines()
    rows = []
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        raw = parts[0]
        img_name = raw[len("img/"):] if raw.startswith("img/") else raw
        rows.append({"image_name": img_name, "item_id": parts[1], "split": parts[2]})
    return rows


def parse_bbox_file(path):
    with open(path) as f:
        lines = f.readlines()
    bboxes = {}
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        raw = parts[0]
        img_name = raw[len("img/"):] if raw.startswith("img/") else raw
        bboxes[img_name] = (int(parts[3]), int(parts[4]), int(parts[5]), int(parts[6]))
    return bboxes


def bbox_crop(img_path, bbox, pad=PAD):
    pil_img = Image.open(img_path).convert("RGB")
    W, H = pil_img.size
    x1, y1, x2, y2 = bbox
    px = int((x2 - x1) * pad); py = int((y2 - y1) * pad)
    x1 = max(0, x1-px); y1 = max(0, y1-py)
    x2 = min(W, x2+px); y2 = min(H, y2+py)
    if x2 <= x1 or y2 <= y1:
        return pil_img
    return pil_img.crop((x1, y1, x2, y2))


# ── Load annotations ──
all_rows  = parse_split_file(SPLIT_FILE)
bbox_map  = parse_bbox_file(BBOX_FILE)
captions  = json.load(open(CAPTIONS_FILE))

gallery_rows = [r for r in all_rows if r["split"] == "gallery"]
query_rows   = [r for r in all_rows if r["split"] == "query"]

print(f"Gallery : {len(gallery_rows)}")
print(f"Query   : {len(query_rows)}")

Gallery : 12612
Query   : 14218


## Cell 4 — FIXED Metric Functions

**Bug 1 fixed:** `n_relevant` now counts actual gallery images sharing the query's item_id, not `len({q_id})` which was always 1.  
**Bug 2 fixed:** `knn_query` now fetches exactly `max_k` results — query images are in the query split, not in gallery, so no self-match can occur.

In [4]:
def recall_at_k(retrieved_ids, relevant_item_id, k):
    """1 if any of top-k retrieved item_ids matches the query item_id."""
    return int(any(iid == relevant_item_id for iid in retrieved_ids[:k]))


def ndcg_at_k(retrieved_ids, relevant_item_id, n_relevant, k):
    """
    NDCG@K.
    n_relevant = number of gallery images with the same item_id as the query.
    IDCG uses min(n_relevant, k) ideal hits at the top positions.
    """
    top_k = retrieved_ids[:k]
    dcg = sum(
        1.0 / np.log2(rank + 2)
        for rank, iid in enumerate(top_k)
        if iid == relevant_item_id
    )
    # FIX: ideal denominator uses actual count of relevant gallery images
    n_ideal = min(n_relevant, k)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(n_ideal))
    return dcg / idcg if idcg > 0 else 0.0


def average_precision_at_k(retrieved_ids, relevant_item_id, n_relevant, k):
    """
    AP@K.
    n_relevant = number of gallery images with the same item_id as the query.
    Denominator = min(n_relevant, k) so AP stays in [0, 1].
    """
    top_k = retrieved_ids[:k]
    hits, prec_sum = 0, 0.0
    for rank, iid in enumerate(top_k, 1):
        if iid == relevant_item_id:
            hits += 1
            prec_sum += hits / rank
    # FIX: actual count of relevant gallery images as denominator
    denom = min(n_relevant, k)
    return prec_sum / denom if denom > 0 else 0.0


def evaluate_retrieval(query_embs, query_ids, gallery_ids, index, k_list=TOP_K_LIST):
    """
    Fixed evaluate_retrieval.
    Builds a per-item_id gallery count map so denominators are correct.
    """
    # FIX: build count of gallery images per item_id once
    gallery_id_to_count = defaultdict(int)
    for iid in gallery_ids:
        gallery_id_to_count[iid] += 1

    max_k = max(k_list)
    gallery_id_arr = np.array(gallery_ids)
    per_query = {f"{m}@{k}": [] for m in ["Recall", "NDCG", "mAP"] for k in k_list}

    # FIX: fetch exactly max_k (queries are not in gallery, no self-match risk)
    labels, _ = index.knn_query(query_embs, k=max_k)

    for q_idx, q_id in enumerate(query_ids):
        retrieved_item_ids = gallery_id_arr[labels[q_idx]].tolist()
        # Actual number of relevant images in the gallery for this query
        n_relevant = gallery_id_to_count.get(q_id, 0)

        for k in k_list:
            per_query[f"Recall@{k}"].append(
                recall_at_k(retrieved_item_ids, q_id, k)
            )
            per_query[f"NDCG@{k}"].append(
                ndcg_at_k(retrieved_item_ids, q_id, n_relevant, k)
            )
            per_query[f"mAP@{k}"].append(
                average_precision_at_k(retrieved_item_ids, q_id, n_relevant, k)
            )

    return {metric: float(np.mean(vals)) for metric, vals in per_query.items()}


print("Fixed metric functions defined.")

Fixed metric functions defined.


## Cell 5 — Load CLIP models & embedding helper
Load both frozen CLIP (for A and B) and fine-tuned CLIP (for C).

In [5]:
# Frozen CLIP — for conditions A and B
clip_frozen, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAIN
)
clip_frozen = clip_frozen.to(DEVICE).eval()

# Fine-tuned CLIP — for condition C
clip_ft, _, _ = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAIN
)
clip_ft.load_state_dict(torch.load(FT_MODEL, map_location="cpu"))
clip_ft = clip_ft.to(DEVICE).eval()

tokenizer = open_clip.get_tokenizer(CLIP_MODEL)
print("Both CLIP models loaded.")


@torch.no_grad()
def generate_embeddings(rows, captions, model, tokenizer, preprocess,
                        bbox_map, img_root, alpha, batch_size=BATCH_SIZE):
    model.eval()
    all_embs, all_ids, all_names = [], [], []
    for i in tqdm(range(0, len(rows), batch_size), desc=f"Embedding (α={alpha})", leave=False):
        batch = rows[i : i + batch_size]
        crops, texts, ids, names = [], [], [], []
        for r in batch:
            img_path = img_root / r["image_name"]
            if not img_path.exists():
                continue
            try:
                bbox = bbox_map.get(r["image_name"])
                crop = bbox_crop(img_path, bbox) if bbox else Image.open(img_path).convert("RGB")
                crops.append(preprocess(crop))
                texts.append(captions.get(r["image_name"], ""))
                ids.append(r["item_id"])
                names.append(r["image_name"])
            except Exception:
                continue
        if not crops:
            continue
        img_t = torch.stack(crops).to(DEVICE)
        txt_t = tokenizer(texts).to(DEVICE)
        img_emb = F.normalize(model.encode_image(img_t).float(), dim=-1)
        txt_emb = F.normalize(model.encode_text(txt_t).float(),  dim=-1)
        fused = (alpha * img_emb + (1.0 - alpha) * txt_emb) if 0 < alpha < 1 else (img_emb if alpha == 1.0 else txt_emb)
        fused = F.normalize(fused, dim=-1)
        all_embs.append(fused.cpu().numpy())
        all_ids.extend(ids)
        all_names.extend(names)
    return np.vstack(all_embs).astype(np.float32), all_ids, all_names


def build_hnsw_index(embeddings, dim=EMB_DIM):
    index = hnswlib.Index(space='cosine', dim=dim)
    index.init_index(max_elements=len(embeddings), ef_construction=200, M=32)
    index.add_items(embeddings, list(range(len(embeddings))))
    index.set_ef(150)
    return index


print("Helpers ready.")

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Both CLIP models loaded.
Helpers ready.


## Cell 6 — Option A: Load saved indexes (fastest)
If you uploaded the `.bin` index files, load them directly — no need to re-embed.  
**Skip to Cell 7 if you did not save the indexes** (will re-embed from images).

In [6]:
def load_index(bin_path, meta_path, dim=EMB_DIM):
    meta = json.load(open(meta_path))
    index = hnswlib.Index(space='cosine', dim=dim)
    index.load_index(str(bin_path))
    index.set_ef(150)
    return index, meta["item_ids"], meta["img_names"]


CONFIGS = [
    # (condition_key, model, alpha, index_bin, meta_json)
    ("A_vision_only_alpha1",       "frozen", 1.0, "gallery_index_A.bin",          "gallery_meta_A.json"),
    ("B_frozen_clip_alpha07",       "frozen", 0.7, "gallery_index_B_alpha07.bin",  "gallery_meta_B_alpha07.json"),
    ("B_frozen_clip_alpha05",       "frozen", 0.5, "gallery_index_B_alpha05.bin",  "gallery_meta_B_alpha05.json"),
    ("C_finetuned_clip_alpha07",    "ft",     0.7, "gallery_index_C_alpha07.bin",  "gallery_meta_C_alpha07.json"),
    ("C_finetuned_clip_alpha05",    "ft",     0.5, "gallery_index_C_alpha05.bin",  "gallery_meta_C_alpha05.json"),
]

all_results = {}

for cond_key, model_type, alpha, bin_name, meta_name in CONFIGS:
    print(f"\n{'='*55}")
    print(f"Condition: {cond_key}")

    bin_path  = SAVED_DIR / bin_name
    meta_path = SAVED_DIR / meta_name
    model     = clip_frozen if model_type == "frozen" else clip_ft

    # Load saved gallery index
    gal_index, gal_ids, gal_names = load_index(bin_path, meta_path)
    print(f"  Gallery index loaded: {len(gal_ids)} images")

    seed_results = []
    for seed in SEEDS:
        set_seed(seed)
        # Only need to re-embed query images (lightweight vs full gallery)
        qry_embs, qry_ids, _ = generate_embeddings(
            query_rows, captions, model, tokenizer, clip_preprocess,
            bbox_map, IMG_ROOT, alpha=alpha
        )
        metrics = evaluate_retrieval(qry_embs, qry_ids, gal_ids, gal_index)
        seed_results.append(metrics)
        print(f"  Seed {seed}: Recall@10={metrics['Recall@10']:.4f}  mAP@10={metrics['mAP@10']:.4f}  NDCG@10={metrics['NDCG@10']:.4f}")

    result = {}
    for metric in seed_results[0]:
        vals = [r[metric] for r in seed_results]
        result[metric] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
    all_results[cond_key] = result

    print(f"  --- Results ({cond_key}) ---")
    for k in TOP_K_LIST:
        for m in ["Recall", "NDCG", "mAP"]:
            mk = f"{m}@{k}"
            print(f"    {mk}: {result[mk]['mean']:.4f} ± {result[mk]['std']:.4f}")

print("\nAll conditions done.")


Condition: A_vision_only_alpha1
  Gallery index loaded: 12612 images


  Seed 83: Recall@10=0.5770  mAP@10=0.1837  NDCG@10=0.2580


  Seed 588: Recall@10=0.5770  mAP@10=0.1837  NDCG@10=0.2580


  Seed 527: Recall@10=0.5770  mAP@10=0.1837  NDCG@10=0.2580


  Seed 33: Recall@10=0.5770  mAP@10=0.1837  NDCG@10=0.2580
  --- Results (A_vision_only_alpha1) ---
    Recall@5: 0.5132 ± 0.0000
    NDCG@5: 0.2564 ± 0.0000
    mAP@5: 0.1935 ± 0.0000
    Recall@10: 0.5770 ± 0.0000
    NDCG@10: 0.2580 ± 0.0000
    mAP@10: 0.1837 ± 0.0000
    Recall@15: 0.6109 ± 0.0000
    NDCG@15: 0.2639 ± 0.0000
    mAP@15: 0.1836 ± 0.0000

Condition: B_frozen_clip_alpha07
  Gallery index loaded: 12612 images


...

  Seed 33: Recall@10=0.8775  mAP@10=0.4673  NDCG@10=0.5660
  --- Results (C_finetuned_clip_alpha07) ---
    Recall@5: 0.8304 ± 0.0000
    NDCG@5: 0.5564 ± 0.0000
    mAP@5: 0.4748 ± 0.0000
    Recall@10: 0.8775 ± 0.0000
    NDCG@10: 0.5660 ± 0.0000
    mAP@10: 0.4673 ± 0.0000
    Recall@15: 0.8984 ± 0.0000
    NDCG@15: 0.5791 ± 0.0000
    mAP@15: 0.4715 ± 0.0000

Condition: C_finetuned_clip_alpha05
  Gallery index loaded: 12612 images


  Seed 83: Recall@10=0.7894  mAP@10=0.3208  NDCG@10=0.4197


  Seed 588: Recall@10=0.7894  mAP@10=0.3208  NDCG@10=0.4197


  Seed 527: Recall@10=0.7894  mAP@10=0.3208  NDCG@10=0.4197


  Seed 33: Recall@10=0.7894  mAP@10=0.3208  NDCG@10=0.4197
  --- Results (C_finetuned_clip_alpha05) ---
    Recall@5: 0.7121 ± 0.0000
    NDCG@5: 0.4039 ± 0.0000
    mAP@5: 0.3229 ± 0.0000
    Recall@10: 0.7894 ± 0.0000
    NDCG@10: 0.4197 ± 0.0000
    mAP@10: 0.3208 ± 0.0000
    Recall@15: 0.8264 ± 0.0000
    NDCG@15: 0.4334 ± 0.0000
    mAP@15: 0.3244 ± 0.0000

All conditions done.


## Cell 7 — Option B: Re-embed gallery from scratch (if no saved indexes)
**Only run this if you did NOT upload the `.bin` index files.**  
Skip entirely if Cell 6 ran successfully.

In [7]:
# Uncomment and run only if index files are unavailable

# all_results = {}
#
# CONFIGS = [
#     ("A_vision_only_alpha1",     clip_frozen, 1.0),
#     ("B_frozen_clip_alpha07",    clip_frozen, 0.7),
#     ("B_frozen_clip_alpha05",    clip_frozen, 0.5),
#     ("C_finetuned_clip_alpha07", clip_ft,     0.7),
#     ("C_finetuned_clip_alpha05", clip_ft,     0.5),
# ]
#
# for cond_key, model, alpha in CONFIGS:
#     print(f"\n{'='*55}\nCondition: {cond_key}")
#     seed_results = []
#     for seed in SEEDS:
#         set_seed(seed)
#         gal_embs, gal_ids, gal_names = generate_embeddings(
#             gallery_rows, captions, model, tokenizer, clip_preprocess,
#             bbox_map, IMG_ROOT, alpha=alpha
#         )
#         qry_embs, qry_ids, _ = generate_embeddings(
#             query_rows, captions, model, tokenizer, clip_preprocess,
#             bbox_map, IMG_ROOT, alpha=alpha
#         )
#         gal_index = build_hnsw_index(gal_embs)
#         metrics = evaluate_retrieval(qry_embs, qry_ids, gal_ids, gal_index)
#         seed_results.append(metrics)
#         print(f"  Seed {seed}: Recall@10={metrics['Recall@10']:.4f}  mAP@10={metrics['mAP@10']:.4f}")
#     result = {}
#     for metric in seed_results[0]:
#         vals = [r[metric] for r in seed_results]
#         result[metric] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
#     all_results[cond_key] = result

print("(Cell 7 is commented out — only run if Cell 6 failed)")

(Cell 7 is commented out — only run if Cell 6 failed)


## Cell 8 — Save corrected metrics & print final table

In [8]:
json.dump(all_results, open(OUTPUT_DIR / "all_metrics_fixed.json", "w"), indent=2)
print(f"Saved → {OUTPUT_DIR / 'all_metrics_fixed.json'}")

print("\n" + "="*80)
print(f"{'Condition':<30} {'R@5':>7} {'R@10':>7} {'R@15':>7} {'mAP@5':>8} {'mAP@10':>8} {'NDCG@10':>9}")
print("="*80)
for cond, res in all_results.items():
    r5   = res['Recall@5']['mean']
    r10  = res['Recall@10']['mean']
    r15  = res['Recall@15']['mean']
    m5   = res['mAP@5']['mean']
    m10  = res['mAP@10']['mean']
    n10  = res['NDCG@10']['mean']
    print(f"{cond:<30} {r5:>7.4f} {r10:>7.4f} {r15:>7.4f} {m5:>8.4f} {m10:>8.4f} {n10:>9.4f}")
print("="*80)

# Sanity check — flag any values still above 1
print("\nSanity check (all values should be ≤ 1.0):")
ok = True
for cond, res in all_results.items():
    for metric, vals in res.items():
        if vals['mean'] > 1.0:
            print(f"  STILL BROKEN: {cond} → {metric} = {vals['mean']:.4f}")
            ok = False
if ok:
    print("  All values ≤ 1.0 ✓")

Saved → /kaggle/working/all_metrics_fixed.json

Condition                          R@5    R@10    R@15    mAP@5   mAP@10   NDCG@10
A_vision_only_alpha1            0.5132  0.5770  0.6109   0.1935   0.1837    0.2580
B_frozen_clip_alpha07           0.3292  0.4097  0.4579   0.0969   0.0938    0.1459
B_frozen_clip_alpha05           0.0148  0.0740  0.1000   0.0026   0.0043    0.0113
C_finetuned_clip_alpha07        0.8304  0.8775  0.8984   0.4748   0.4673    0.5660
C_finetuned_clip_alpha05        0.7121  0.7894  0.8264   0.3229   0.3208    0.4197

Sanity check (all values should be ≤ 1.0):
  All values ≤ 1.0 ✓
